# IRRA baseline and Gallery Context on RSTPReid

Attach Kaggle dataset `hoanggv/tbps-benchmark`, enable GPU (T4 x2 works; this simple run uses one GPU), then run all cells. The repository contains the copied IRRA implementation plus an optional Gallery Context module.

In [ ]:
REPO_URL = "https://github.com/lqb464/gallery-conditioned-tbps.git"
!if [ -d /kaggle/working/gallery-conditioned-tbps/.git ]; then git -C /kaggle/working/gallery-conditioned-tbps pull --ff-only; else git clone -q $REPO_URL /kaggle/working/gallery-conditioned-tbps; fi
%cd /kaggle/working/gallery-conditioned-tbps
!pip -q install -r requirements.txt

In [ ]:
import torch, os
print(torch.cuda.get_device_name(0))
DATA_ROOT = "/kaggle/input/datasets/hoanggv/tbps-benchmark"
OUT = "/kaggle/working/irra_gallery_runs"
os.makedirs(OUT, exist_ok=True)

## Train original IRRA baseline

This keeps the Gallery Context module disabled and provides a comparable same-data baseline.

In [ ]:
!python train.py --dataset_name RSTPReid --root_dir "$DATA_ROOT" --output_dir "$OUT" --name irra_baseline --val_dataset val --loss_names sdm+id --sampler identity --num_instance 4 --batch_size 32 --test_batch_size 256 --num_workers 2 --num_epoch 60 --img_aug --lr 1e-5 --weight_decay 4e-5 --warmup_epochs 5 --lrscheduler cosine --eval_period 1

## Train Gallery Context

Same IRRA backbone, data pipeline, and losses; the only architectural change is self-attention over each training batch and over the full validation/test image gallery.

In [ ]:
!python train.py --dataset_name RSTPReid --root_dir "$DATA_ROOT" --output_dir "$OUT" --name gallery_context --val_dataset val --loss_names sdm+id --sampler identity --num_instance 4 --batch_size 32 --test_batch_size 256 --num_workers 2 --num_epoch 60 --img_aug --lr 1e-5 --weight_decay 4e-5 --warmup_epochs 5 --lrscheduler cosine --eval_period 1 --gallery_context --gallery_context_depth 2 --gallery_context_heads 8

## Final test evaluation

Run each evaluation against its best validation checkpoint.

In [ ]:
!find "$OUT/RSTPReid" -name best.pth -print

The commands below locate each run's saved config and evaluate the best validation checkpoint.

In [ ]:
from pathlib import Path
configs = list(Path(OUT, "RSTPReid").glob("*/configs.yaml"))
for config in configs:
    print("Evaluating", config)
    !python test.py --config_file "$config"